## 1. Model 불러오기

In [1]:
import torch
import torch.nn.functional as F
from torchvision import models
from torchvision.models.quantization import mobilenet_v2
from torch.ao.quantization import get_default_qat_qconfig, prepare_qat
from torchvision.models import MobileNet_V2_Weights
from tqdm import tqdm
from torchvision.models.quantization import mobilenet_v2
from torch.ao.quantization import get_default_qconfig, prepare, convert, QuantStub, DeQuantStub

In [2]:
class MobileNetV2_MixedPTQ(torch.nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.stem = backbone.features[0] # FP32
        self.quant = QuantStub()        # FP32 → INT8
        self.features = backbone.features[1:]
        self.dequant = DeQuantStub()    # INT8 → FP32
        self.classifier = backbone.classifier

    def forward(self, x):
        x = self.stem(x)
        x = self.quant(x)               # FP32 -> INT8
        x = self.features(x)            # INT8 backbone
        x = self.dequant(x)             # 경계 명시

        x = F.adaptive_avg_pool2d(x, 1)
        x = torch.flatten(x, 1)

        x = self.classifier(x)          # FP32 classifier
        return x

In [4]:
model_fp32 = models.mobilenet_v2(
    weights=MobileNet_V2_Weights.IMAGENET1K_V1
)

backbone = mobilenet_v2(quantize=False)
backbone.load_state_dict(model_fp32.state_dict())
backbone.fuse_model()

model_quantizable = MobileNetV2_MixedPTQ(backbone)

torch.backends.quantized.engine = "fbgemm"
model_quantizable.qconfig = get_default_qat_qconfig("fbgemm")
model_quantizable.stem.qconfig = None
model_quantizable.classifier.qconfig = None

model_quantizable.train()
model_qat = prepare_qat(model_quantizable, inplace=False)

# QAT weight 로드
state_dict = torch.load(
    "./model/mobilenetv2_mixed_qat_epoch8_70.12.pth",
    map_location="cpu"
)
model_qat.load_state_dict(state_dict, strict=False)

# 🔥 convert
model_int8 = convert(model_qat.eval(), inplace=False)
model_int8.eval()

/Users/Kpiano/miniconda3/envs/qat/lib/python3.10/site-packages/torch/ao/quantization/utils.py:339: UserWarning: must run observer before calling calculate_qparams. Returning default values.
  warnings.warn(


MobileNetV2_MixedPTQ(
  (stem): Conv2dNormActivation(
    (0): ConvBnReLU2d(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
    )
    (1): Identity()
    (2): Identity()
  )
  (quant): Quantize(scale=tensor([0.0199]), zero_point=tensor([0]), dtype=torch.quint8)
  (features): Sequential(
    (1): QuantizableInvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): QuantizedConvReLU2d(32, 32, kernel_size=(3, 3), stride=(1, 1), scale=0.0885215774178505, zero_point=0, padding=(1, 1), groups=32)
          (1): Identity()
          (2): Identity()
        )
        (1): QuantizedConv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), scale=0.09448595345020294, zero_point=65)
        (2): Identity()
      )
      (skip_add): QFunctional(
        scale=1.0, zero_point=0
        (activation_post_process): Identity()
 

## 2. Layer 수집

In [18]:
# IC = 960 Layer 찾기
import torch.nn.quantized as nnq

target_layers = []

for name, m in model_int8.named_modules():
    if isinstance(m, nnq.Conv2d):
        if m.weight().shape[1] == 960:
            target_layers.append((name, m))

print(target_layers)

[('features.15.conv.2', QuantizedConv2d(960, 160, kernel_size=(1, 1), stride=(1, 1), scale=0.01223889458924532, zero_point=59)), ('features.16.conv.2', QuantizedConv2d(960, 160, kernel_size=(1, 1), stride=(1, 1), scale=0.019143395125865936, zero_point=63)), ('features.17.conv.2', QuantizedConv2d(960, 320, kernel_size=(1, 1), stride=(1, 1), scale=0.019200945273041725, zero_point=63))]


In [47]:
# Pointwise Layer 전체 수집
target_layers = []

for name, m in model_int8.named_modules():
    if isinstance(m, torch.nn.quantized.Conv2d):
        if m.groups == 1:   # depthwise 제외
            target_layers.append((name, m))

target_layers

[('features.1.conv.1',
  QuantizedConv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), scale=0.09448595345020294, zero_point=65)),
 ('features.2.conv.0.0',
  QuantizedConvReLU2d(16, 96, kernel_size=(1, 1), stride=(1, 1), scale=0.046150870621204376, zero_point=0)),
 ('features.2.conv.2',
  QuantizedConv2d(96, 24, kernel_size=(1, 1), stride=(1, 1), scale=0.05688821151852608, zero_point=57)),
 ('features.3.conv.0.0',
  QuantizedConvReLU2d(24, 144, kernel_size=(1, 1), stride=(1, 1), scale=0.01197422482073307, zero_point=0)),
 ('features.3.conv.2',
  QuantizedConv2d(144, 24, kernel_size=(1, 1), stride=(1, 1), scale=0.06257268041372299, zero_point=61)),
 ('features.4.conv.0.0',
  QuantizedConvReLU2d(24, 144, kernel_size=(1, 1), stride=(1, 1), scale=0.01743791066110134, zero_point=0)),
 ('features.4.conv.2',
  QuantizedConv2d(144, 32, kernel_size=(1, 1), stride=(1, 1), scale=0.044303443282842636, zero_point=68)),
 ('features.5.conv.0.0',
  QuantizedConvReLU2d(32, 192, kernel_size=(1, 1), stride=

In [48]:
# Weight INT8 추출
for name, layer in target_layers:
    w_int = layer.weight().int_repr()  # 🔥 실제 int8 값
    print(name, w_int.shape)

features.1.conv.1 torch.Size([16, 32, 1, 1])
features.2.conv.0.0 torch.Size([96, 16, 1, 1])
features.2.conv.2 torch.Size([24, 96, 1, 1])
features.3.conv.0.0 torch.Size([144, 24, 1, 1])
features.3.conv.2 torch.Size([24, 144, 1, 1])
features.4.conv.0.0 torch.Size([144, 24, 1, 1])
features.4.conv.2 torch.Size([32, 144, 1, 1])
features.5.conv.0.0 torch.Size([192, 32, 1, 1])
features.5.conv.2 torch.Size([32, 192, 1, 1])
features.6.conv.0.0 torch.Size([192, 32, 1, 1])
features.6.conv.2 torch.Size([32, 192, 1, 1])
features.7.conv.0.0 torch.Size([192, 32, 1, 1])
features.7.conv.2 torch.Size([64, 192, 1, 1])
features.8.conv.0.0 torch.Size([384, 64, 1, 1])
features.8.conv.2 torch.Size([64, 384, 1, 1])
features.9.conv.0.0 torch.Size([384, 64, 1, 1])
features.9.conv.2 torch.Size([64, 384, 1, 1])
features.10.conv.0.0 torch.Size([384, 64, 1, 1])
features.10.conv.2 torch.Size([64, 384, 1, 1])
features.11.conv.0.0 torch.Size([384, 64, 1, 1])
features.11.conv.2 torch.Size([96, 384, 1, 1])
features.12.c

In [49]:
activations = {}

def hook_fn(name):
    def hook(module, input, output):
        act_q = input[0]  # quantized tensor
        activations[name] = {
            "int": act_q.int_repr(),
            "zp": act_q.q_zero_point()
        }
    return hook

for name, layer in target_layers:
    layer.register_forward_hook(hook_fn(name))

## 3. Data 불러오기

In [50]:
import glob

def iter_subset(dir_path):
    files = sorted(glob.glob(f"{dir_path}/*.pt"))
    for f in files:
        yield torch.load(f)

In [51]:
ds_calib = iter_subset("imagenet_subset/calib")
ds_qat   = iter_subset("imagenet_subset/qat_train")
ds_val   = iter_subset("imagenet_subset/val")

In [52]:
ds_calib

<generator object iter_subset at 0x12a8be500>

## 4. Maximum 값 탐색

In [53]:
INT24_MAX = 2**23 - 1

layer_max_dict = {}
global_max = 0
global_layer = None

for sample in ds_val:

    img = sample["image"].unsqueeze(0)
    model_int8(img)

    for name, layer in target_layers:

        act_info = activations[name]
        act_int = act_info["int"].to(torch.int32)
        act_zp  = act_info["zp"]

        w_int = layer.weight().int_repr().to(torch.int32)
        act_int = act_int - act_zp

        N, IC, H, W = act_int.shape
        OC = w_int.shape[0]

        act_flat = act_int.permute(0,2,3,1).reshape(-1, IC)
        weight_flat = w_int.view(OC, IC).t()

        psum = act_flat @ weight_flat
        max_abs = psum.abs().max().item()

        # layer별 max 저장
        if name not in layer_max_dict:
            layer_max_dict[name] = 0
        if max_abs > layer_max_dict[name]:
            layer_max_dict[name] = max_abs

        # global max
        if max_abs > global_max:
            global_max = max_abs
            global_layer = name

In [54]:
print("=== Layer-wise Max PSUM ===")
for name, val in sorted(layer_max_dict.items(), key=lambda x: -x[1]):
    print(f"{name:30s} : {val}")

print("\n=== Global ===")
print("Global max:", global_max)
print("Occurred at:", global_layer)
print("Margin:", INT24_MAX - global_max)

=== Layer-wise Max PSUM ===
features.18.0                  : 329575
features.9.conv.2              : 220065
features.16.conv.2             : 219144
features.13.conv.2             : 199034
features.12.conv.2             : 193649
features.8.conv.2              : 193294
features.11.conv.2             : 162953
features.10.conv.2             : 135683
features.3.conv.2              : 121708
features.15.conv.2             : 115758
features.17.conv.2             : 108740
features.7.conv.2              : 99211
features.5.conv.2              : 83544
features.6.conv.2              : 83077
features.14.conv.2             : 65948
features.4.conv.2              : 63253
features.13.conv.0.0           : 50352
features.14.conv.0.0           : 50044
features.12.conv.0.0           : 45890
features.2.conv.2              : 45716
features.16.conv.0.0           : 45526
features.17.conv.0.0           : 44921
features.8.conv.0.0            : 38802
features.11.conv.0.0           : 36571
features.9.conv.0.0      